# Data Science for Business - Interpretable ML for Predicting House Prices

*Session 12 · Beyond ISLP · Hooks: ISLP 2.1.3 (accuracy vs. interpretability), 8.2.1 (variable importance measures, Fig. 8.9), 7.7 + Lab 7.8.3 (GAM partial dependence) · External: Molnar, Interpretable Machine Learning; Lundberg & Lee (2017)*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
# Install packages that are not pre-installed on Colab
# !pip install shap

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.inspection import permutation_importance
from sklearn.inspection import PartialDependenceDisplay

import shap

In [ ]:
np.random.seed(42)
plt.style.use('fivethirtyeight')

## Problem description

Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence. With 76 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, this dataset challenges you to predict the final price of each home. More: <https://www.kaggle.com/c/house-prices-advanced-regression-techniques>

Flexible models such as random forests often predict better than linear regression, but they are much harder to interpret (ISLP 2.1.3). In this notebook we fit a random forest and then use model-agnostic tools to look inside it: feature importance, partial dependence plots (PDP), individual conditional expectation (ICE) plots and SHAP values.

ISLP covers interpretability only briefly (2.1.3, 8.2.1). Further reading: C. Molnar, *Interpretable Machine Learning* (<https://christophm.github.io/interpretable-ml-book/>), chapters on PDP, ICE, permutation feature importance and SHAP.

## Load data

Load the Ames housing data (2,930 houses) from a CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_12/ameshousing.csv')
data.shape

In [ ]:
data.head()

## Prepare data

Let us first focus on some easy to understand variables.

In [ ]:
data = data[["SalePrice", "LotArea", "GrLivArea", "FullBath", "BedroomAbvGr", "KitchenAbvGr", "OverallQual", "OverallCond"]]

In [ ]:
data.head()

Finally, we will split the data into predictors (*X*) and response (*y*) and into training (*X_train, y_train*) and test (*X_test, y_test*) sets (validation set approach, ISLP 5.1.1).

All predictors are stored as integers. We convert them to floats, because scikit-learn's partial dependence plots (used below) do not accept integer columns.

In [ ]:
X = data.drop("SalePrice", axis=1).astype(float)
y = data["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Fit a random forest model

Since we are mainly interested in interpreting the model, we will skip feature engineering and hyperparameter tuning and directly fit a random forest (ISLP 8.2.2) to the training data.

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

In [ ]:
pred = rf_model.predict(X_test)
rmse = root_mean_squared_error(y_test, pred)
print(rmse)

The RMSE is not super great, but that is not the main focus of this analysis, as we are interested in interpreting the model. Still, as we can see below, the random forest performs better than linear regression.

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
pred = lr_model.predict(X_test)
rmse = root_mean_squared_error(y_test, pred)
print(rmse)

## Global explanations: Feature importance

Which predictors matter most for the model overall? We compare two measures:

- **Impurity-based importance** (ISLP 8.2.1, *Variable importance measures*, Fig. 8.9): the total decrease in RSS due to splits on a predictor, averaged over all trees. It comes for free with tree ensembles (`feature_importances_`), but it is computed on the training data and biased toward predictors with many distinct values.
- **Permutation importance** (not in ISLP): shuffle one predictor in the **test** data and measure how much the test error worsens. It is model-agnostic and measured in the units of the metric (here: dollars of RMSE).

With correlated predictors (e.g., `GrLivArea`, `FullBath`, `BedroomAbvGr`), both measures can split the credit between them.

### Impurity-based importance

In [ ]:
impurity_importances = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
impurity_importances

### Permutation importance

Use `permutation_importance` as a model-agnostic function to calculate feature importance on the test set. Each predictor is shuffled 10 times.

In [ ]:
r = permutation_importance(rf_model, X_test, y_test, scoring="neg_root_mean_squared_error", n_repeats=10, random_state=42)

Put the results into a dataframe for easier manipulation and plotting, and sort it by the mean importance.

In [ ]:
importances = pd.DataFrame({'feature': X_test.columns, 'value': r.importances_mean, 'std': r.importances_std})
importances = importances.sort_values(by='value', ascending=False)
importances

Plot the results as a bar plot. The black lines show the standard deviation across the 10 repetitions.

In [ ]:
sns.barplot(data=importances, x='value', y='feature')
plt.errorbar(data=importances, x='value', y='feature', xerr='std', fmt='none', color='black')
plt.xlabel('Increase in RMSE after permutation')
plt.ylabel('Feature')
plt.show()

### Comparison

Side by side, both measures agree on the most important predictors but can differ in the ranking of the less important ones.

In [ ]:
pd.DataFrame({'impurity': impurity_importances, 'permutation (RMSE increase)': importances.set_index('feature')['value']}).sort_values(by='impurity', ascending=False)

## Global explanations: Partial dependence plots (PDP)

Next, we explore the partial dependence of the response on the predictors. A PDP sets one predictor to a value $x$ for **all** houses and averages the predictions: $\hat f_S(x) = \frac{1}{n}\sum_{i=1}^{n} \hat f(x, x_{i,C})$, where $x_{i,C}$ are the other predictors of house $i$ (Friedman's partial dependence).

This differs from ISLP's "partial dependence" plots for GAMs (Lab 7.8.3), which show a GAM component function while the other variables are fixed at their **mean**. For an additive model like a GAM both views coincide (up to a constant); for a random forest with interactions they can differ.

The tick marks at the bottom show where the data lie; beyond them we extrapolate.

In [ ]:
features = ["LotArea"]
PartialDependenceDisplay.from_estimator(rf_model, X_test, features=features)
plt.show()

In [ ]:
features = ["GrLivArea"]
PartialDependenceDisplay.from_estimator(rf_model, X_test, features=features)
plt.show()

We can also create 2D partial dependence plots to explore possible interactions between predictors.

In [ ]:
features = [("LotArea", "GrLivArea")]
PartialDependenceDisplay.from_estimator(rf_model, X_test, features=features)
plt.show()

Here is an example of a non-linear partial dependence between a predictor and the response: the predicted price rises with `OverallCond` up to an average condition (5) and then levels off.

In [ ]:
features = ["OverallCond"]
PartialDependenceDisplay.from_estimator(rf_model, X_test, features=features)
plt.show()

## Local explanations: Individual conditional expectation (ICE) plots

ICE plots are the equivalent of partial dependence plots for individual observations. Each thin line is one house; the thick line is the PDP (their average). Parallel lines indicate an additive effect; lines with different shapes or crossing lines indicate interactions with other predictors.

In [ ]:
features = ["LotArea"]
PartialDependenceDisplay.from_estimator(rf_model, X_test, features=features, kind='both')
plt.show()

## Local explanations: SHAP values

Finally, we will calculate SHAP values (Lundberg & Lee, 2017) to explain single predictions from the test set. SHAP values split a prediction into additive contributions of the predictors, relative to the average prediction over a **background** dataset. As we have fitted a random forest, we will use the fast `TreeExplainer`. There are other explainers for other types of learners (e.g., neural networks).

As background data we use a random sample of 100 houses from the training set.

In [ ]:
background = shap.sample(X_train, 100, random_state=42)
explainer = shap.TreeExplainer(rf_model, background)

Let's focus on one house in the test set (the house in row 42).

Here are its features:

In [ ]:
obs = 42
X_test.iloc[[obs], :]

The true value (response):

In [ ]:
y_test.iloc[obs]

And the predicted value:

In [ ]:
rf_model.predict(X_test.iloc[[obs], :])[0]

Using the `explainer` object initialized above, we will now calculate SHAP values for all houses in the test set (to be able to compare the predicted price of an individual house with the average prediction).

In [ ]:
shap_values = explainer(X_test)

Next, we plot the computed SHAP values for our house as a `waterfall` plot.

In [ ]:
shap.plots.waterfall(shap_values[obs])

Read the plot bottom-up: start at $E[f(X)]$, the average prediction over the background data, add each predictor's SHAP value (red pushes the price up, blue pushes it down) and arrive at this house's prediction $f(x)$, in dollars.

SHAP values can also be aggregated into a global view. In the `beeswarm` plot, each dot is one house in the test set; the position shows the SHAP value and the color the value of the predictor (red = high, blue = low). Predictors are sorted by their mean absolute SHAP value.

In [ ]:
shap.plots.beeswarm(shap_values)

The excellent `SHAP` package (<https://shap.readthedocs.io>) provides many more plots to visualize individual SHAP values and to aggregate local SHAP values into global explanations.

## Your turn!

**Your turn!** Explain the most expensive house in the test set. Which predictors drive its predicted price? (Hint: `y_test.argmax()` gives its row position.)

In [ ]:
# YOUR CODE HERE

**Your turn!** Draw a PDP and an ICE plot for `OverallQual`. Is its effect additive, or do the ICE lines suggest interactions?

In [ ]:
# YOUR CODE HERE